## 1. Del workflow visual de n8n al notebook en Python

Antes de escribir la lógica principal del notebook, es importante entender qué estamos replicando.

En n8n construimos un workflow visual compuesto por varias cajas. Cada caja tenía una responsabilidad concreta dentro del proceso:

```text
Manual Trigger
        ↓
Prompt - Analizar candidato
        ↓
Azure Foundry - Analizar CV
        ↓
Resultado limpio
        ↓
Generar informe descargable
```

La idea era sencilla: ejecutar manualmente el flujo, preparar un prompt con información de un candidato, enviarlo a un modelo desplegado en Azure AI Foundry, limpiar la respuesta y generar un informe descargable.

En este notebook vamos a implementar el mismo concepto, pero con código Python. La diferencia principal es que ahora el candidato no estará escrito manualmente en el prompt, sino que vendrá desde un archivo PDF.

---

### Equivalencia entre n8n y el notebook

| Workflow en n8n             | Equivalente en Python                       |
| --------------------------- | ------------------------------------------- |
| Manual Trigger              | Ejecutar las celdas del notebook            |
| Prompt - Analizar candidato | Construir un prompt dinámico en Python      |
| Azure Foundry - Analizar CV | Hacer una petición HTTP con `requests`      |
| Resultado limpio            | Extraer el texto útil de la respuesta JSON  |
| Generar informe descargable | Mostrar o guardar el resultado como informe |

---

### Qué problema resolvemos

En procesos de selección reales, los equipos de Recursos Humanos pueden recibir muchos currículums en formato PDF. Revisarlos uno por uno puede ser lento y repetitivo.

Este notebook simula una primera capa de automatización que permite:

* Leer el contenido de un CV en PDF.
* Extraer el texto del documento.
* Enviar ese texto a un modelo de IA.
* Analizar skills, experiencia y carencias.
* Generar preguntas recomendadas para una entrevista.
* Proponer una decisión inicial sobre el candidato.

No sustituye a una persona de Recursos Humanos, pero puede ayudar a acelerar la primera revisión y organizar mejor la información.

---

### Arquitectura lógica del notebook

El proceso que vamos a seguir será:

```text
1. Cargar configuración desde .env
2. Leer el archivo PDF del candidato
3. Extraer el texto del PDF
4. Crear un prompt especializado para RRHH
5. Enviar el prompt a Azure AI Foundry
6. Recibir la respuesta del modelo
7. Mostrar el análisis final en formato legible
```

---

### Por qué usamos un archivo `.env`

En el workflow de n8n, la API Key se configuraba dentro del nodo HTTP. En código, no es buena práctica escribir claves directamente en el notebook.

Por eso utilizamos un archivo `.env`, donde guardamos:

```text
AZURE_AI_ENDPOINT
AZURE_AI_API_KEY
AZURE_AI_MODEL
```

De esta forma, el notebook puede leer la configuración sin exponer secretos dentro del código principal.

---

### Patrón técnico que estamos aplicando

Este notebook sigue un patrón muy habitual en proyectos de IA generativa:

```text
Documento no estructurado
        ↓
Extracción de texto
        ↓
Prompt engineering
        ↓
Modelo de lenguaje
        ↓
Respuesta estructurada
        ↓
Decisión de negocio
```

En nuestro caso, el documento no estructurado es un CV en PDF, y la decisión de negocio es una primera recomendación sobre si el candidato debería avanzar en el proceso de selección.

---

### Resultado esperado

Al final del notebook esperamos obtener un análisis parecido a este:

```text
RESUMEN DEL CANDIDATO:
Descripción general del perfil profesional.

SKILLS DETECTADAS:
Tecnologías, herramientas y competencias encontradas en el CV.

NIVEL DE ENCAJE:
Alto / Medio / Bajo

GAPS DETECTADOS:
Aspectos que faltan o que deberían validarse.

PREGUNTAS RECOMENDADAS PARA ENTREVISTA:
Preguntas adaptadas al perfil del candidato.

DECISIÓN RECOMENDADA:
Avanzar, revisar manualmente o descartar.
```

Esta salida replica el resultado que ya conseguimos en n8n, pero ahora usando un PDF como entrada real del proceso.


In [4]:
# ============================================================
# 1. Configuración inicial del notebook
# ============================================================
#
# Esta celda replica la parte de configuración que en n8n teníamos repartida
# entre el nodo HTTP Request y los parámetros del workflow.
#
# Aquí cargamos:
# - Endpoint de Azure AI Foundry
# - API Key
# - Nombre del modelo/deployment
#
# Todo se lee desde el archivo .env para no dejar secretos escritos en el notebook.

import os
import requests

from dotenv import load_dotenv
from pypdf import PdfReader
from IPython.display import Markdown, display

# ------------------------------------------------------------
# Cargar variables desde .env
# ------------------------------------------------------------

load_dotenv()

AZURE_AI_ENDPOINT = os.getenv("AZURE_AI_ENDPOINT")
AZURE_AI_API_KEY = os.getenv("AZURE_AI_API_KEY")
AZURE_AI_MODEL = os.getenv("AZURE_AI_MODEL")

# ------------------------------------------------------------
# Validar que la configuración existe
# ------------------------------------------------------------

required_variables = {
    "AZURE_AI_ENDPOINT": AZURE_AI_ENDPOINT,
    "AZURE_AI_API_KEY": AZURE_AI_API_KEY,
    "AZURE_AI_MODEL": AZURE_AI_MODEL,
}

missing_variables = [
    variable_name
    for variable_name, variable_value in required_variables.items()
    if not variable_value
]

if missing_variables:
    raise ValueError(
        "Faltan variables en el archivo .env: "
        + ", ".join(missing_variables)
    )

# ------------------------------------------------------------
# Confirmación visual
# ------------------------------------------------------------

print("Configuración cargada correctamente.")
print(f"Endpoint configurado: {AZURE_AI_ENDPOINT}")
print(f"Modelo/Deployment configurado: {AZURE_AI_MODEL}")
print("API Key cargada correctamente: sí")

Configuración cargada correctamente.
Endpoint configurado: https://ia-adnan.services.ai.azure.com/api/projects/proj-default/openai/v1/responses
Modelo/Deployment configurado: gpt-4.1
API Key cargada correctamente: sí


## 2. Creación de un agente simple con Microsoft Agent Framework (MAF)

Para cumplir explícitamente el punto de la entrega sobre creación de agentes, en esta sección creamos un agente mínimo con **Microsoft Agent Framework** conectado a Azure AI Foundry.

Este agente es sencillo: recibe una pregunta de prueba y responde usando el modelo desplegado en Foundry. Después, el resto del notebook aplica la misma idea al caso de uso del análisis de CV.

In [5]:
# Si no tienes estas dependencias instaladas, ejecuta esta celda una vez.
# !pip install agent-framework azure-identity

# MAF usa AzureCliCredential en este ejemplo.
# Si no has iniciado sesión en Azure, ejecuta:
# !az login

In [6]:
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


# El endpoint de MAF debe ser el endpoint base del proyecto/recurso Foundry,
# no la URL completa terminada en /openai/v1/responses.
FOUNDRY_PROJECT_ENDPOINT = "https://ia-adnan.services.ai.azure.com/"
FOUNDRY_MODEL = AZURE_AI_MODEL

foundry_client = FoundryChatClient(
    project_endpoint=FOUNDRY_PROJECT_ENDPOINT,
    model=FOUNDRY_MODEL,
    credential=AzureCliCredential(),
)

maf_agent = Agent(
    client=foundry_client,
    name="AgenteSimpleMAF",
    instructions=(
        "Eres un agente creado con Microsoft Agent Framework. "
        "Responde siempre en español, de forma breve y clara."
    ),
)

print("Agente MAF creado correctamente:", maf_agent.name)

c:\Users\adnan\Desktop\agente-linkedin\.venv\lib\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Users\adnan\Desktop\agente-linkedin\.venv\lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


Agente MAF creado correctamente: AgenteSimpleMAF


In [7]:
# Prueba mínima del agente MAF.
# Esta celda llama al modelo de Azure AI Foundry usando Microsoft Agent Framework.

maf_result = await maf_agent.run(
    "Explica en una frase cuál es tu función como agente de preselección de CVs."
)

print("Respuesta del agente MAF:")
print(maf_result.text)

Respuesta del agente MAF:
Mi función es analizar y evaluar currículums para identificar candidatos que cumplen con los requisitos del puesto y recomendar a los más adecuados para continuar en el proceso de selección.


## 3. Lectura del CV en PDF y extracción de texto

Una vez cargada la configuración del proyecto desde el archivo `.env`, el siguiente paso es preparar la entrada real del sistema.

En el workflow de n8n, el candidato estaba escrito directamente dentro de una caja de tipo `Set / Edit Fields`. Es decir, el texto del candidato estaba “mockeado” manualmente.

En este notebook vamos a mejorar ese enfoque usando un archivo PDF como entrada.

---

### Qué haremos en esta fase

En esta sección vamos a:

```text id="dig5ta"
1. Indicar la ruta del archivo PDF del candidato.
2. Abrir el PDF desde Python.
3. Recorrer sus páginas.
4. Extraer el texto disponible.
5. Validar que realmente se ha extraído contenido.
```

---

### Por qué extraemos texto del PDF

Los modelos de lenguaje no analizan directamente un PDF como archivo binario en esta implementación sencilla.

Primero necesitamos convertir el contenido del documento a texto plano. Ese texto será el que después insertaremos dentro del prompt enviado a Azure AI Foundry.

El flujo queda así:

```text id="7di9ol"
Archivo PDF
    ↓
Texto extraído
    ↓
Prompt para el modelo
    ↓
Análisis del candidato
```

---

### Limitación importante

Este método funciona bien con PDFs que contienen texto real, por ejemplo un CV exportado desde Word, Google Docs, Canva o LinkedIn.

Sin embargo, si el PDF es una imagen escaneada, puede que no se extraiga texto correctamente. En ese caso haría falta usar OCR, por ejemplo con Azure Document Intelligence, Tesseract u otra herramienta de reconocimiento óptico de caracteres.

Para esta demo académica usaremos un PDF con texto legible para mantener el notebook simple y fácil de explicar.

---

### Resultado esperado de la siguiente celda

La siguiente celda de código deberá devolver algo parecido a esto:

```text id="0zvj4s"
PDF cargado correctamente.
Número de páginas: 1
Caracteres extraídos: 2450
```

Con eso sabremos que el contenido del CV ya está listo para ser enviado al modelo de IA.


In [8]:
# ============================================================
# 2. Lectura del CV en PDF y extracción de texto
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Ruta del PDF
# ------------------------------------------------------------
# Coloca aquí el nombre de tu archivo PDF.
# El PDF debe estar en la misma carpeta que el notebook.
#
# Ejemplo:
# PDF_PATH = "cv_candidato.pdf"

PDF_PATH = "Cv_Avanade_Hamidoun_Adnan.pdf"

pdf_path = Path(PDF_PATH)

if not pdf_path.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo PDF: {PDF_PATH}\n"
        "Asegúrate de que el PDF está en la misma carpeta que el notebook "
        "o cambia la ruta en la variable PDF_PATH."
    )

# ------------------------------------------------------------
# Abrir el PDF y extraer texto página por página
# ------------------------------------------------------------

reader = PdfReader(str(pdf_path))

texto_paginas = []

for page_number, page in enumerate(reader.pages, start=1):
    texto_pagina = page.extract_text() or ""
    texto_paginas.append(texto_pagina)

# Unimos todo el texto en una sola variable
cv_text = "\n\n".join(texto_paginas).strip()

# ------------------------------------------------------------
# Validación básica
# ------------------------------------------------------------

if not cv_text:
    raise ValueError(
        "No se ha podido extraer texto del PDF. "
        "Puede que el archivo sea un PDF escaneado como imagen y necesite OCR."
    )

print("PDF cargado correctamente.")
print(f"Número de páginas: {len(reader.pages)}")
print(f"Caracteres extraídos: {len(cv_text)}")

# Mostramos una vista previa del contenido extraído
print("\nVista previa del texto extraído:")
print("-" * 60)
print(cv_text[:1000])

PDF cargado correctamente.
Número de páginas: 1
Caracteres extraídos: 4524

Vista previa del texto extraído:
------------------------------------------------------------
C O N T A C T O
H A B I L I D A D E S  P E R S O N A L E S
I D I O M A S
C E R T I F I C A C I O N E S  O F I C I A L E S
h t t p s : / / w w w . l i n k e d i n . c o m / i n / a d n a
n - h a m i d o u n
O r i e n t a c i ó n  a  r e s u l t a d o s
I n f l u e n c i a  y  C o m u n i c a c i ó n
T o m a  d e  d e c i s i o n e s
A p e r t u r a  y  A d a p t a b i l i d a d
C o n f i a n z a  y  S e g u r i d a d
I n g l é s :  B 2  
E s p a ñ o l :  N a t i v o
Á r a b e :  N a t i v o
A D N A N  H A M I D O U N  E L  H A B T I
D A T A  &  A I  S O L U T I O N S  E N G I N E E R  |  M I C R O S O F T  F A B R I C  &  A Z U R E  |  B U S I N E S S  &  T E C H
E X P E R I E N C I A
E D U C A C I Ó N
S O B R E  M I
T i t u l a d o  e n  D A M  y  e s p e c i a l i z a d o  e n  B i g  D a t a  e  I n t e l i g e n c i

## 3. Validación de la extracción del PDF

En la celda anterior hemos cargado el archivo PDF del candidato y hemos extraído su contenido en formato texto.

Este paso es importante porque confirma que el notebook ya no depende de un candidato escrito manualmente, sino de un documento real como entrada del sistema.

---

### Qué hemos comprobado

La celda anterior valida tres cosas:

```text id="qthyt1"
1. Que el archivo PDF existe en la ruta indicada.
2. Que Python puede abrir el documento correctamente.
3. Que el PDF contiene texto extraíble.
```

Si cualquiera de estas partes falla, el notebook muestra un error claro para saber qué corregir.

---

### Por qué esta fase es clave

El modelo de IA no recibe directamente el PDF como archivo, sino el texto que hemos extraído de él.

Por eso, antes de llamar a Azure AI Foundry, necesitamos asegurarnos de que la variable `cv_text` contiene información útil del candidato.

El flujo en este punto queda así:

```text id="n8bipo"
CV en PDF
    ↓
Extracción con PyPDF
    ↓
Variable cv_text
    ↓
Prompt dinámico
    ↓
Azure AI Foundry
```

---

### Qué significa que la extracción haya funcionado

Si la celda ha mostrado un resultado parecido a este:

```text id="k4dw2c"
PDF cargado correctamente.
Número de páginas: 1
Caracteres extraídos: 2450
```

significa que el PDF se ha leído correctamente y que ya tenemos texto suficiente para construir el prompt.

Además, la vista previa nos permite revisar rápidamente si el contenido extraído tiene sentido: nombre, experiencia, estudios, tecnologías, proyectos o cualquier información relevante del CV.

---

### Limitación técnica

Este método funciona especialmente bien con PDFs digitales, es decir, documentos que contienen texto real.

Por ejemplo:

```text id="yv67p5"
- CV exportado desde Word
- CV exportado desde Google Docs
- CV descargado desde LinkedIn
- CV creado con Canva, siempre que conserve texto seleccionable
```

Si el PDF fuera un escaneo o una imagen, PyPDF probablemente no podría extraer texto. En ese caso sería necesario añadir una fase de OCR.

Para esta versión del notebook mantenemos la solución sencilla y directa, igual que en el workflow de n8n, pero sustituyendo el texto fijo por una entrada documental real.

---

### Siguiente paso

Ahora que ya tenemos el contenido del CV dentro de `cv_text`, el siguiente paso será construir un prompt profesional de Recursos Humanos.

Ese prompt combinará:

```text id="53m0zi"
- Instrucciones del rol del agente
- Formato esperado de la respuesta
- Texto completo extraído del PDF
```

Con esto conseguiremos que el modelo analice el CV y devuelva una salida estructurada.


In [9]:
# ============================================================
# 3. Construcción del prompt dinámico para Azure AI Foundry
# ============================================================
#
# En n8n teníamos un texto fijo dentro del nodo "Prompt - Analizar candidato".
# Ahora ese texto se genera dinámicamente a partir del contenido real del PDF.

# ------------------------------------------------------------
# Control básico de tamaño
# ------------------------------------------------------------
# Para una demo sencilla, limitamos el texto enviado al modelo.
# Esto evita problemas si el PDF es muy largo.

MAX_CV_CHARS = 12000

cv_text_limited = cv_text[:MAX_CV_CHARS]

if len(cv_text) > MAX_CV_CHARS:
    print(f"El CV es largo. Se enviarán solo los primeros {MAX_CV_CHARS} caracteres.")
else:
    print("El CV completo cabe dentro del límite configurado.")

# ------------------------------------------------------------
# Prompt profesional para análisis de candidato
# ------------------------------------------------------------

user_prompt = f"""
Actúa como un asistente de Recursos Humanos especializado en perfiles tecnológicos.

Tu tarea es analizar el siguiente CV extraído desde un archivo PDF.

Debes devolver una evaluación clara, profesional y estructurada.

Analiza:
- Experiencia profesional
- Skills técnicas
- Herramientas y tecnologías mencionadas
- Nivel de encaje general
- Puntos fuertes
- Gaps o carencias
- Preguntas recomendadas para entrevista
- Decisión recomendada

Formato obligatorio de respuesta:

RESUMEN DEL CANDIDATO:
[Resumen profesional del perfil]

SKILLS DETECTADAS:
[Lista de tecnologías, herramientas y competencias]

NIVEL DE ENCAJE:
[Alto / Medio / Bajo]

PUNTOS FUERTES:
[Lista de fortalezas principales]

GAPS DETECTADOS:
[Lista de carencias o aspectos a validar]

PREGUNTAS RECOMENDADAS PARA ENTREVISTA:
[Preguntas adaptadas al perfil]

DECISIÓN RECOMENDADA:
[Avanzar / Revisar manualmente / Descartar, justificando brevemente]

CV EXTRAÍDO DEL PDF:
\"\"\"
{cv_text_limited}
\"\"\"
"""

print("Prompt construido correctamente.")
print(f"Caracteres del CV enviados al modelo: {len(cv_text_limited)}")
print(f"Caracteres totales del prompt: {len(user_prompt)}")

print("\nVista previa del prompt:")
print("-" * 60)
print(user_prompt[:1500])

El CV completo cabe dentro del límite configurado.
Prompt construido correctamente.
Caracteres del CV enviados al modelo: 4524
Caracteres totales del prompt: 5476

Vista previa del prompt:
------------------------------------------------------------

Actúa como un asistente de Recursos Humanos especializado en perfiles tecnológicos.

Tu tarea es analizar el siguiente CV extraído desde un archivo PDF.

Debes devolver una evaluación clara, profesional y estructurada.

Analiza:
- Experiencia profesional
- Skills técnicas
- Herramientas y tecnologías mencionadas
- Nivel de encaje general
- Puntos fuertes
- Gaps o carencias
- Preguntas recomendadas para entrevista
- Decisión recomendada

Formato obligatorio de respuesta:

RESUMEN DEL CANDIDATO:
[Resumen profesional del perfil]

SKILLS DETECTADAS:
[Lista de tecnologías, herramientas y competencias]

NIVEL DE ENCAJE:
[Alto / Medio / Bajo]

PUNTOS FUERTES:
[Lista de fortalezas principales]

GAPS DETECTADOS:
[Lista de carencias o aspectos a val

## 4. Construcción del prompt dinámico

En la celda anterior hemos construido el prompt que se enviará al modelo de Azure AI Foundry.

Este paso es el equivalente directo al nodo de n8n llamado:

```text
Prompt - Analizar candidato
```

La diferencia es que en n8n el texto del candidato estaba escrito manualmente, mientras que ahora el prompt se construye automáticamente usando el contenido real extraído del PDF.

---

### Qué hace esta celda

La celda de código anterior genera una variable llamada:

```python
user_prompt
```

Esta variable contiene tres partes principales:

```text
1. El rol que debe adoptar el modelo.
2. Las instrucciones de análisis.
3. El texto real extraído del CV en PDF.
```

---

### Rol del modelo

Primero indicamos al modelo cómo debe comportarse:

```text
Actúa como un asistente de Recursos Humanos especializado en perfiles tecnológicos.
```

Esto es importante porque orienta la respuesta hacia un contexto concreto. No queremos una respuesta genérica, sino un análisis útil para un proceso de selección.

---

### Instrucciones de análisis

Después se le indica al modelo qué debe revisar dentro del CV:

```text
- Experiencia profesional
- Skills técnicas
- Herramientas y tecnologías mencionadas
- Nivel de encaje general
- Puntos fuertes
- Gaps o carencias
- Preguntas recomendadas para entrevista
- Decisión recomendada
```

De esta forma, el modelo no se limita a resumir el documento, sino que realiza una evaluación estructurada.

---

### Formato obligatorio de salida

También se define el formato exacto que debe devolver el modelo:

```text
RESUMEN DEL CANDIDATO:
SKILLS DETECTADAS:
NIVEL DE ENCAJE:
PUNTOS FUERTES:
GAPS DETECTADOS:
PREGUNTAS RECOMENDADAS PARA ENTREVISTA:
DECISIÓN RECOMENDADA:
```

Este formato nos permite obtener una respuesta limpia, fácil de leer y similar a la salida que generamos en n8n.

---

### Control de tamaño del CV

La celda también incluye un límite de caracteres:

```python
MAX_CV_CHARS = 12000
```

Esto sirve para evitar enviar un documento demasiado largo al modelo.

En una versión más avanzada, podríamos dividir el CV en fragmentos o aplicar una estrategia de resumen por partes. Para esta demo, limitar el tamaño es suficiente y mantiene el notebook sencillo.

---

### Flujo actual del notebook

Hasta este punto, el notebook ya ha completado las siguientes fases:

```text
1. Cargar configuración desde .env
2. Leer el CV en PDF
3. Extraer el texto del PDF
4. Construir un prompt dinámico con el contenido real del CV
```

El siguiente paso será enviar este prompt al endpoint de Azure AI Foundry usando una petición HTTP, igual que hacía el nodo HTTP Request en n8n.


In [10]:
# ============================================================
# 4. Llamada al endpoint de Azure AI Foundry
# ============================================================
#
# Esta celda replica el nodo HTTP Request de n8n.
# Enviamos el prompt dinámico al modelo desplegado en Azure AI Foundry
# y recibimos una respuesta en formato JSON.

# ------------------------------------------------------------
# Cabeceras de la petición
# ------------------------------------------------------------

headers = {
    "Content-Type": "application/json",
    "api-key": AZURE_AI_API_KEY
}

# ------------------------------------------------------------
# Cuerpo de la petición
# ------------------------------------------------------------
# Importante:
# - El campo "model" usa el nombre del deployment/modelo configurado en .env
# - El endpoint /responses espera "input"
# - Cada mensaje debe llevar "type": "message"

payload = {
    "model": AZURE_AI_MODEL,
    "input": [
        {
            "type": "message",
            "role": "system",
            "content": (
                "Eres un agente experto en selección de perfiles tecnológicos. "
                "Responde siempre en español, con formato claro, profesional y estructurado."
            )
        },
        {
            "type": "message",
            "role": "user",
            "content": user_prompt
        }
    ],
    "temperature": 0.4,
    "max_output_tokens": 700,
    "text": {
        "format": {
            "type": "text"
        },
        "verbosity": "medium"
    }
}

# ------------------------------------------------------------
# Enviar petición
# ------------------------------------------------------------

response = requests.post(
    AZURE_AI_ENDPOINT,
    headers=headers,
    json=payload,
    timeout=60
)

# ------------------------------------------------------------
# Validar respuesta HTTP
# ------------------------------------------------------------

if response.status_code != 200:
    print("Error en la llamada a Azure AI Foundry")
    print(f"Status code: {response.status_code}")
    print(response.text)
    response.raise_for_status()

azure_response = response.json()

print("Respuesta recibida correctamente desde Azure AI Foundry.")
print(f"ID de respuesta: {azure_response.get('id')}")
print(f"Estado: {azure_response.get('status')}")

Respuesta recibida correctamente desde Azure AI Foundry.
ID de respuesta: resp_05ae57f15c1f001b006a213a5e80248197a370e674baabdf1f
Estado: incomplete


In [11]:
azure_response["output"][0]["content"][0]["text"]

'RESUMEN DEL CANDIDATO:\nProfesional especializado en ingeniería de datos y soluciones de inteligencia artificial, con experiencia en arquitecturas escalables, procesamiento de grandes volúmenes de datos y desarrollo de aplicaciones multiplataforma. Destaca por su enfoque en aportar valor real al negocio mediante soluciones tecnológicas robustas y rentables. Posee formación en Big Data, Inteligencia Artificial y desarrollo de aplicaciones, así como certificaciones oficiales en el ecosistema Microsoft Azure y metodologías ágiles.\n\nSKILLS DETECTADAS:\n- Microsoft Fabric (Data & AI Solutions)\n- Azure (Data Fundamentals, DB Administrator, Analytics Engineer, Data Engineer, Data Scientist, AI Engineer)\n- PySpark\n- MySQL\n- Databricks (AI Agent Fundamentals, Fundamentals)\n- Confluent Data Streaming\n- Power BI (visualización de datos)\n- Machine Learning (AutoML)\n- MLOps\n- Desarrollo de aplicaciones multiplataforma\n- Gestión de errores y excepciones\n- Metodología ágil (Scrum Fundam

## 5. Recepción de la respuesta de Azure AI Foundry

En la celda anterior hemos enviado el prompt dinámico al endpoint de Azure AI Foundry.

Ese paso es el equivalente al nodo de n8n:

```text
Azure Foundry - Analizar CV
```

En n8n, este nodo devolvía un objeto JSON grande con mucha información técnica: identificador de la respuesta, estado, modelo utilizado, tokens consumidos, filtros de seguridad y contenido generado por el modelo.

En el notebook ocurre lo mismo. La respuesta no llega directamente como texto plano, sino como una estructura JSON.

---

### Qué contiene la respuesta

La respuesta de Azure AI Foundry incluye información como:

```text
- ID de la respuesta
- Estado de ejecución
- Modelo utilizado
- Texto generado por la IA
- Uso de tokens
- Metadatos técnicos
```

Para nuestro caso de uso, no necesitamos mostrar todo el JSON al usuario final.

Lo más importante es extraer únicamente el análisis generado por el modelo.

---

### Ruta del texto generado

En nuestro workflow de n8n, limpiamos la salida usando una expresión parecida a:

```text
{{$json.output[0].content[0].text}}
```

En Python haremos lo mismo, pero accediendo al diccionario de respuesta:

```python
azure_response["output"][0]["content"][0]["text"]
```

Esa ruta nos permite recuperar solamente el texto útil del análisis.

---

### Objetivo de la siguiente celda

La siguiente celda hará tres cosas:

```text
1. Extraer el análisis final desde la respuesta JSON.
2. Mostrarlo de forma legible en el notebook.
3. Prepararlo para poder guardarlo como informe.
```

Con esto completamos la misma lógica que teníamos en n8n con el nodo:

```text
Resultado limpio
```

Es decir, pasamos de una respuesta técnica de API a una salida clara para negocio.


In [12]:
# ============================================================
# 5. Extracción y visualización de la respuesta limpia
# ============================================================
#
# Esta celda replica el nodo "Resultado limpio" de n8n.
# Extraemos únicamente el texto generado por la IA desde el JSON completo.

# ------------------------------------------------------------
# Extraer texto útil de la respuesta
# ------------------------------------------------------------

try:
    analysis_text = azure_response["output"][0]["content"][0]["text"]
except (KeyError, IndexError, TypeError) as error:
    print("No se pudo extraer el texto de la respuesta de Azure.")
    print("Estructura recibida:")
    print(azure_response)
    raise error

# ------------------------------------------------------------
# Mostrar información técnica básica
# ------------------------------------------------------------

response_id = azure_response.get("id", "No disponible")
status = azure_response.get("status", "No disponible")
model_used = azure_response.get("model", AZURE_AI_MODEL)

usage = azure_response.get("usage", {})
total_tokens = usage.get("total_tokens", "No disponible")

print("Análisis extraído correctamente.")
print(f"ID de respuesta: {response_id}")
print(f"Estado: {status}")
print(f"Modelo utilizado: {model_used}")
print(f"Tokens totales: {total_tokens}")

# ------------------------------------------------------------
# Mostrar el análisis en formato Markdown
# ------------------------------------------------------------

display(Markdown("## Informe generado por IA"))
display(Markdown(analysis_text))

Análisis extraído correctamente.
ID de respuesta: resp_05ae57f15c1f001b006a213a5e80248197a370e674baabdf1f
Estado: incomplete
Modelo utilizado: gpt-4.1
Tokens totales: 3477


## Informe generado por IA

RESUMEN DEL CANDIDATO:
Profesional especializado en ingeniería de datos y soluciones de inteligencia artificial, con experiencia en arquitecturas escalables, procesamiento de grandes volúmenes de datos y desarrollo de aplicaciones multiplataforma. Destaca por su enfoque en aportar valor real al negocio mediante soluciones tecnológicas robustas y rentables. Posee formación en Big Data, Inteligencia Artificial y desarrollo de aplicaciones, así como certificaciones oficiales en el ecosistema Microsoft Azure y metodologías ágiles.

SKILLS DETECTADAS:
- Microsoft Fabric (Data & AI Solutions)
- Azure (Data Fundamentals, DB Administrator, Analytics Engineer, Data Engineer, Data Scientist, AI Engineer)
- PySpark
- MySQL
- Databricks (AI Agent Fundamentals, Fundamentals)
- Confluent Data Streaming
- Power BI (visualización de datos)
- Machine Learning (AutoML)
- MLOps
- Desarrollo de aplicaciones multiplataforma
- Gestión de errores y excepciones
- Metodología ágil (Scrum Fundamentals)
- Business Analysis
- Idiomas: Español (nativo), Árabe (nativo), Inglés (B2)

NIVEL DE ENCAJE:
Alto

PUNTOS FUERTES:
- Amplia formación y certificaciones en tecnologías Microsoft Azure y Big Data.
- Experiencia en construcción de arquitecturas escalables y seguras.
- Capacidad para implementar soluciones end-to-end en Data & AI.
- Competencia en visualización y automatización de flujos de datos (Power BI, MLOps).
- Experiencia en integración con servicios externos y optimización de procesos financieros.
- Habilidades personales orientadas a resultados, toma de decisiones y adaptabilidad.
- Multilingüe, con nivel B2 de inglés.

GAPS DETECTADOS:
- No se especifica experiencia en entornos cloud distintos de Azure (por ejemplo, AWS o GCP).
- Falta de detalle sobre proyectos reales en producción y su impacto en negocio.
- No se menciona experiencia en liderazgo de equipos o gestión de proyectos.
- Nivel de inglés B2, podría ser insuficiente para ciertos entornos internacionales.
- No se detallan frameworks de desarrollo backend/frontend ni lenguajes adicionales a PySpark/MySQL.

PREGUNTAS RECOMENDADAS PARA ENTREVISTA:
1. ¿Puedes describir algún proyecto donde hayas implementado una arquitectura de datos end-to-end en producción? ¿Cuál fue el impacto en el negocio?
2. ¿Qué retos has enfrentado al procesar grandes volúmenes de datos y cómo los resolviste?
3. ¿Has trabajado con otros proveedores cloud además de Azure? ¿Cómo compararías tu experiencia?
4. ¿Qué metodología utilizas para asegurar la calidad y seguridad de las soluciones desarrolladas?
5. ¿Cómo gestionas la integración y comunicación con servicios externos en proyectos complejos?
6. ¿Tienes experiencia liderando equipos o gestionando proyectos? ¿Podrías dar ejemplos?
7. ¿Cómo te mantienes actualizado en tecnologías emergentes de Data & AI?
8. ¿Cuál es tu nivel de autonomía en inglés para reuniones técnicas o documentación?

DECISIÓN RECOMENDADA:
Avanzar. El candidato presenta un perfil sólido y alineado con posiciones de Data & AI Engineer, especialmente en entornos Microsoft Azure. Se recomienda profundizar en la entrevista sobre proyectos reales, capacidad de liderazgo y nivel de inglés, pero su formación, certificaciones y experiencia técnica

## 6. Generación de un informe descargable

En este punto el notebook ya ha completado el flujo principal:

```text
CV en PDF
    ↓
Extracción de texto
    ↓
Prompt dinámico
    ↓
Azure AI Foundry
    ↓
Análisis estructurado
```

Sin embargo, para que el resultado sea más útil, podemos guardar el análisis generado por la IA en un archivo Markdown.

Esto replica la última caja del workflow de n8n:

```text
Generar informe descargable
```

---

### Por qué guardar el informe

Mostrar el resultado en pantalla está bien para una demo, pero en un caso real sería útil poder guardar o compartir el análisis.

Por ejemplo, el informe podría enviarse a:

```text
- Un técnico de selección
- Un responsable de equipo
- Un sistema interno de RRHH
- Una carpeta documental del proceso
```

En esta versión sencilla, guardaremos el informe como archivo `.md`.

---

### Qué contendrá el informe

El archivo generado incluirá:

```text
- Título del informe
- Fecha de generación
- Nombre del modelo utilizado
- Análisis completo generado por la IA
- Información técnica básica del flujo
```

---

### Resultado esperado

La siguiente celda creará un archivo llamado:

```text
informe_candidato.md
```

Este archivo podrá abrirse directamente en VS Code, GitHub, Obsidian, Word o cualquier visor compatible con Markdown.

Si se quisiera convertir a PDF, se podría abrir el Markdown y usar:

```text
Imprimir → Guardar como PDF
```

Con esto cerramos el ciclo completo: desde un CV en PDF hasta un informe final reutilizable.


In [13]:
# ============================================================
# 6. Generación de informe descargable en Markdown
# ============================================================
#
# Esta celda replica el nodo final de n8n:
# "Generar informe descargable"
#
# En lugar de dejar el análisis solo en pantalla, lo guardamos
# como un archivo .md reutilizable.

from datetime import datetime
from pathlib import Path

# ------------------------------------------------------------
# Configuración del archivo de salida
# ------------------------------------------------------------

output_file = Path("informe_candidato.md")

fecha_generacion = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

# ------------------------------------------------------------
# Construcción del contenido del informe
# ------------------------------------------------------------

report_content = f"""# Informe de Preselección de Candidato

**Generado automáticamente por notebook Python + Azure AI Foundry**  
**Fecha de generación:** {fecha_generacion}  
**Modelo/Deployment utilizado:** {AZURE_AI_MODEL}  
**Estado de la respuesta:** {status}  
**Tokens totales:** {total_tokens}

---

## Análisis generado por IA

{analysis_text}

---

## Información técnica del proceso

Este informe ha sido generado mediante un flujo automatizado que realiza los siguientes pasos:

1. Carga la configuración desde un archivo `.env`.
2. Lee un currículum en formato PDF.
3. Extrae el texto del documento usando Python.
4. Construye un prompt dinámico para análisis de perfiles tecnológicos.
5. Envía el prompt a un modelo desplegado en Azure AI Foundry.
6. Extrae la respuesta limpia del modelo.
7. Guarda el resultado como informe Markdown.

---

## Caso de uso

Este notebook simula un agente de preselección de candidatos tecnológicos.

El objetivo no es sustituir la revisión humana, sino acelerar una primera evaluación del perfil y ayudar a estructurar la información antes de una entrevista técnica.
"""

# ------------------------------------------------------------
# Guardar archivo
# ------------------------------------------------------------

output_file.write_text(report_content, encoding="utf-8")

print("Informe generado correctamente.")
print(f"Archivo creado: {output_file.resolve()}")

Informe generado correctamente.
Archivo creado: C:\Users\adnan\Desktop\agente-linkedin\informe_candidato.md
